# C) Training: RegNetY-1.6GF + freezing `lp_ft` (ricetta `acq_mild`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR       = Path('/content/dataset_local')
SPLIT_CSV         = BASE / 'splits' / 'split.csv'
MODELS_DIR        = BASE / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE    = 224
NUM_CLASSES = 8
SEED        = 1234
EPOCHS      = 15
RECIPE_TAG  = 'acq_mild'

CFG_NAME     = 'lp_ft'
FREEZE_ATTRS = []                                                               # attributi da congelare all'inizio ([] = full fine-tune)
LP_EPOCHS    = 3

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

OUT_PATH = MODELS_DIR / f"regnety16gf_{RECIPE_TAG}_{CFG_NAME}.pth"
print('Config:', CFG_NAME, '| freeze:', FREEZE_ATTRS, '| lp_epochs:', LP_EPOCHS)
print('Output atteso:', OUT_PATH)

Mounted at /content/drive
Config: lp_ft | freeze: [] | lp_epochs: 3
Output atteso: /content/drive/MyDrive/2026_MLinf_gr41/Waste-Project/models/regnety16gf_acq_mild_lp_ft.pth


In [ ]:
import io, os, random, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '|', torch.cuda.get_device_name(0) if device.type=='cuda' else 'CPU')

if not DATASET_DIR.exists():
    print('Copio il dataset in locale...')
    t0 = time.time(); shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Fatto in {time.time()-t0:.0f}s')
else:
    print('Copia locale gia presente.')

Device: cuda | Tesla T4
Copio il dataset in locale...
Fatto in 582s


## Split e augmentation `acq_mild`

In [ ]:
df_all = pd.read_csv(SPLIT_CSV)
df_train = df_all[df_all['split'] == 'train'].reset_index(drop=True).copy()
df_val   = df_all[df_all['split'] == 'val'].reset_index(drop=True).copy()
print(f"Train: {len(df_train)}  |  Val: {len(df_val)}")

class RandomJPEG:
    def __init__(self, quality_min=50, quality_max=95, p=0.5):
        self.quality_min, self.quality_max, self.p = quality_min, quality_max, p
    def __call__(self, img):
        if random.random() < self.p:
            q = random.randint(self.quality_min, self.quality_max)
            buf = io.BytesIO(); img.save(buf, format='JPEG', quality=q); buf.seek(0)
            img = Image.open(buf).convert('RGB')
        return img

class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.5):
        self.std, self.p = std, p
    def __call__(self, t):
        if random.random() < self.p:
            t = (t + torch.randn_like(t) * self.std).clamp(0.0, 1.0)
        return t

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.3),
    RandomJPEG(quality_min=50, quality_max=95, p=0.5),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.5),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

Train: 12413  |  Val: 3102


## Backbone con freezing + dataset + eval

In [ ]:
def build_frozen_regnety(freeze_attrs):
    m = models.regnet_y_1_6gf(weights='DEFAULT')
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    for attr in freeze_attrs:
        module = m.stem if attr == 'stem' else getattr(m.trunk_output, attr)
        for p in module.parameters():
            p.requires_grad = False
    n_total  = sum(p.numel() for p in m.parameters())
    n_frozen = sum(p.numel() for p in m.parameters() if not p.requires_grad)
    print(f"  Parametri: {n_total/1e6:.2f}M totali, {n_frozen/1e6:.2f}M congelati "
          f"({100*n_frozen/n_total:.1f}%)")
    return m

class TrainDataset(Dataset):
    def __init__(self, df): self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        return train_tf(Image.open(DATASET_DIR / fp).convert('RGB')), lab

class ValDataset(Dataset):
    def __init__(self, df):
        self.items = list(zip(df['filepath'].tolist(), df['label'].tolist()))
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        fp, lab = self.items[i]
        return preprocess(Image.open(DATASET_DIR / fp).convert('RGB')), lab

@torch.no_grad()
def evaluate_clean(model, df, bs=64):
    model.eval()
    dl = DataLoader(ValDataset(df), batch_size=bs, shuffle=False, num_workers=2)
    ys, ps = [], []
    for x, y in dl:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy()); ys.append(np.asarray(y))
    return balanced_accuracy_score(np.concatenate(ys), np.concatenate(ps))

## Training loop

In [ ]:
def train():
    set_seed(SEED)
    model = build_frozen_regnety(FREEZE_ATTRS if LP_EPOCHS == 0 else []).to(device)

    if LP_EPOCHS > 0:
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith('fc.')

    opt = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=0.01, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit  = nn.CrossEntropyLoss()
    micro, accum = 32, 2

    train_dl = DataLoader(TrainDataset(df_train), batch_size=micro, shuffle=True,
                          num_workers=2, drop_last=True)
    best_bal, best_state, peak_mem = -1.0, None, 0.0

    for epoch in range(EPOCHS):
        if LP_EPOCHS > 0 and epoch == LP_EPOCHS:
            for p in model.parameters():
                p.requires_grad = True
            opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS - LP_EPOCHS)
            print(f"  epoca {epoch+1}: sblocco tutti i layer (full fine-tune)")

        model.train()
        if epoch == 0 and device.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        opt.zero_grad()
        for i, (x, y) in enumerate(tqdm(train_dl, desc=f"{CFG_NAME} ep{epoch+1}/{EPOCHS}", leave=False)):
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y) / accum
            loss.backward()
            if (i + 1) % accum == 0:
                opt.step(); opt.zero_grad()
        if len(train_dl) % accum != 0:
            opt.step(); opt.zero_grad()
        sched.step()
        if epoch == 0 and device.type == 'cuda':
            peak_mem = torch.cuda.max_memory_allocated() / 1e9
        bal = evaluate_clean(model, df_val)
        if bal > best_bal:
            best_bal = bal
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  epoch {epoch+1:2d}: val balAcc = {bal:.4f}  (best {best_bal:.4f})")

    torch.save(best_state, OUT_PATH)
    print(f"\n-> salvato {OUT_PATH.name} | best val balAcc {best_bal:.4f} | train peak {peak_mem:.2f} GB")
    del model; torch.cuda.empty_cache()
    return best_bal, peak_mem

In [ ]:
if OUT_PATH.exists():
    print(f"[skip] {OUT_PATH.name} gia presente sul Drive.")
else:
    best_bal, peak_mem = train()
    print(f"\nFatto.")